In [1]:
!pip install -q -U transformers accelerate bitsandbytes datasets peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 41.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [2]:
# 1. Install all necessary libraries
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# 2. Authenticate Hugging Face
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(hf_token)

In [3]:
# 2. Model & Tokenizer
model_id = "Qwen/Qwen2.5-Math-1.5B-Instruct"

In [4]:


tokenizer = AutoTokenizer.from_pretrained(model_id)
# Ensure pad_token is set to avoid the warning you saw earlier
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16, 
    device_map={"": 0}, 
    low_cpu_mem_usage=True
)

# 3. Dataset (Orca-Math)
## 1. Load the L3 dataset
raw_dataset = load_dataset(
    "openbmb/UltraData-Math", 
    "UltraData-Math-L3-QA-Synthetic", 
    split="train[:10000]"
)

##2. Refined Mapping for 'content' column
def split_ultra_data(sample):
    text_content = sample['content']
    
    # Check if the dataset already contains our desired headers
    if "Question:" in text_content and "Answer:" in text_content:
        # If it's already formatted, we just ensure it matches your specific style
        return {"text": text_content.replace("Question:", "### Question:").replace("Answer:", "### Answer:")}
    
    # Fallback: If it's a single block, we treat the whole thing as a solution 
    # and provide a generic header to maintain the training format.
    return {"text": f"### Question:\nSolve the following mathematical problem.\n\n### Answer:\n{text_content}"}

dataset = raw_dataset.map(split_ultra_data)


# 4. Training Config (Tuning for stability)
training_args = SFTConfig(
    output_dir="/kaggle/working/math-solver-1.5b",
    dataset_text_field="text",
    max_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)
peft_config = LoraConfig(
    r=32,             # Increased from 16 for better capacity
    lora_alpha=64,    # Increased from 32
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM"
)

# 5. Execute
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_args,
)

print("Starting Fine-Tuning...")
trainer.train()

# Save final
trainer.save_model("/kaggle/working/math-solver-final-1.5B")

config.json:   0%|          | 0.00/656 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Tokenizing train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting Fine-Tuning...


Step,Training Loss
10,0.572438
20,0.520845
30,0.469980
40,0.561236
50,0.486873
60,0.472699
70,0.379038
80,0.470107
90,0.393507
100,0.463749
